# 06.08 · Hybrid search, query rewriting, and re-ranking

**Module 06 · Documents, Embeddings & RAG** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 120–150 min |
| 📋 Prerequisites | Lessons 06.04–06.06. |
| 🔑 Keys | `GROQ_API_KEY` (optional), `COHERE_API_KEY` (optional, paid comparison) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Explain** what keyword search (BM25) finds that vector search misses, and vice versa.
2. **Fuse** both rankings with reciprocal-rank fusion (RRF): hybrid search.
3. **Rewrite** a question into several search queries (multi-query), built explicitly instead of `MultiQueryRetriever`.
4. **Re-rank** candidates with a local cross-encoder (free; Cohere optional).
5. **Measure** every addition on the golden and paraphrase sets, and keep only what helps.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Retrieval: hybrid RAG](https://docs.langchain.com/oss/python/deepagents/retrieval) | Query enhancement, retrieval validation, answer checks |
| [FastEmbed docs: rerankers](https://qdrant.github.io/fastembed/) | `TextCrossEncoder` models (ONNX, CPU) |
| [Cohere reranker (LangChain)](https://docs.langchain.com/oss/python/integrations/retrievers/cohere-reranker) | `CohereRerank` (paid API, trial key available) |
| [Agentic RAG with LangGraph: rewrite the question](https://docs.langchain.com/oss/python/langgraph/agentic-rag) | Query rewriting inside a graph (Module 07) |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "langchain-text-splitters>=1.1.3,<2" "pypdf>=6.20.1,<7" "beautifulsoup4>=4.15.0,<5" "fastembed>=0.9.0,<1"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langchain-core", "langchain-text-splitters",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/06-rag/).
import importlib.util
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY', 'COHERE_API_KEY')
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Step 4: Silence known, harmless noise so real problems stand out. Set these BEFORE importing chromadb/fastembed.
import os
import warnings

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma: no usage telemetry.
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")   # Windows: HF cache can't symlink without Developer Mode.
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")        # HF tokenizers' fork warning in Jupyter.
# Same as Lesson 02.04: langchain-core 1.6.9/1.6.10 Pydantic serializer warning about the injected runtime.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# tqdm (used by the model downloader) asks for ipywidgets to draw a progress bar; text output is fine.
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
# Step 5: The course corpus: 24 Markdown files + 8 PDFs (23 pages) about fictional Nimbus weather stations.
import json

RAG_DIR = REPO_ROOT / "data" / "samples" / "rag"
CORPUS_DIR = RAG_DIR / "corpus"
GOLDEN = [json.loads(line) for line in (RAG_DIR / "golden-set.jsonl").read_text(encoding="utf-8").splitlines()]
print("corpus files:", sum(1 for p in CORPUS_DIR.rglob("*") if p.is_file()), "| golden questions:", len(GOLDEN))

> 🌐 First run downloads a small model from Hugging Face (no key, no tokens), cached afterwards. Offline, blocked, or `[embeddings]` not installed? The cell prints a ⚠️ and the lesson continues with the offline `HashingEmbeddings`.

In [ ]:
# Step 6 (🌐 first run only): chunks, vector index, BM25 index, and the paraphrase set from 06.04.
from mastery.config import is_set
from mastery.rag import (BM25, build_index, default_embeddings, embedding_label, evaluate_retrieval, hybrid_search,
                         load_corpus, retrieve, rrf, split_documents)

chunks = split_documents(load_corpus(CORPUS_DIR), "markdown", 800, 100)
embeddings = default_embeddings()
store = build_index(chunks, embeddings)
bm25 = BM25(chunks)
PAIRS = [json.loads(line) for line in (RAG_DIR / "embedding-pairs.jsonl").read_text(encoding="utf-8").splitlines()]
PARAPHRASE_SET = [{"id": f"p{i:02d}", "question": p["question"], "expected_sources": [p["expected_source"]],
                   "in_corpus": True} for i, p in enumerate(PAIRS, start=1)]
HAVE_COHERE_KEY = is_set("COHERE_API_KEY")
print(len(chunks), "chunks |", embedding_label(embeddings), "| BM25 vocabulary:", len(bm25._idf), "terms")

**You should now see** `73 chunks | BAAI/bge-small-en-v1.5 | BM25 vocabulary: ~700 terms`.

---
## Part 1 · Two kinds of search

- **Vector search** compares meaning: great for paraphrases, weak on exact identifiers (error codes, part numbers, names) that the model never learned.
- **BM25** (the classic search-engine score) counts shared words, weighting rare ones higher: great for `E40` or `rev B`, blind to synonyms.

In [ ]:
# 🧪 Offline: one exact-code query and one paraphrase, both ways.
def show(label: str, docs) -> None:
    print(f"  {label:<7}", [d.metadata["source"].split("/")[-1] for d in docs])

for q in ["E40", "My account got blocked after typing the wrong password several times"]:
    print(q)
    show("vector", retrieve(store, q, k=3))
    show("bm25", bm25.search(q, k=3))

**You should now see** both methods put `nimbus-pro-manual.pdf` first for the bare code `E40`, but notice the difference: BM25 returns **only one** result (the only chunk that contains `E40`), while vector search fills the list with loosely related product pages. For the password paraphrase both find `faq-app-login.md` first; BM25's next hits are noise that happens to share words ("several", "times"). On bigger corpora with many similar codes the gap on identifiers grows quickly. Our `BM25` is ~40 lines of pure Python in `mastery/rag/retrieve.py` (LangChain's `BM25Retriever` needs the extra `rank_bm25` package and lives in `langchain-community`, which is no longer maintained).

---
## Part 2 · Hybrid search with reciprocal-rank fusion

BM25 scores and cosine similarities live on different scales, so you can't just add them. **RRF** uses only ranks: each list gives a document `1 / (60 + rank)`, and the sums decide. A document ranked well by **both** wins.

In [ ]:
# 🧪 Offline: RRF by hand on tiny lists, then on real results.
from langchain_core.documents import Document

a, b, c = (Document(t, metadata={"chunk_id": t}) for t in "abc")
print("fused:", [(d.metadata["chunk_id"], d.metadata["score"]) for d in rrf([a, b, c], [b, c])])

q = "What should I do about error E40?"
show("vector", retrieve(store, q, k=3))
show("bm25", bm25.search(q, k=3))
show("hybrid", hybrid_search(store, bm25, q, k=3))

**You should now see** `fused: [('b', 0.0325), ('c', 0.0320), ('a', 0.0164)]` (b and c appear in both lists, a only in one), then the three rankings for the E40 question, with the Pro manual at or near the top of the hybrid list.

Now measure all three on both question sets:

In [ ]:
# 🧪/🌐 Measure: recall@1, recall@5 and MRR for vector, BM25 and hybrid.
SEARCHES = {
    "vector": lambda q, k: retrieve(store, q, k=k),
    "bm25": lambda q, k: bm25.search(q, k=k),
    "hybrid": lambda q, k: hybrid_search(store, bm25, q, k=k),
}

def report(searches: dict) -> None:
    print(f"{'method':<16}{'gold R@1':>9}{'gold MRR':>9}{'para R@1':>9}{'para R@5':>9}{'para MRR':>9}")
    for name, search in searches.items():
        g1 = evaluate_retrieval(lambda q: search(q, 1), GOLDEN, k=1)
        g5 = evaluate_retrieval(lambda q: search(q, 5), GOLDEN, k=5)
        p1 = evaluate_retrieval(lambda q: search(q, 1), PARAPHRASE_SET, k=1)
        p5 = evaluate_retrieval(lambda q: search(q, 5), PARAPHRASE_SET, k=5)
        print(f"{name:<16}{g1['recall']:>9.2f}{g5['mrr']:>9.2f}{p1['recall']:>9.2f}{p5['recall']:>9.2f}{p5['mrr']:>9.2f}")

report(SEARCHES)

**You should now see** three rows. On our run with bge-small: vector and hybrid tie on the golden set (≈ 0.91 R@1), BM25 is weaker on paraphrases, and **hybrid lifts paraphrase R@1 from 0.80 to 0.90**. Small sets → small, noisy differences; the direction is what to look at. (With `HashingEmbeddings` the "vector" row is itself word-based, so hybrid adds little.)

---
## Part 3 · Query rewriting (multi-query)

Users ask vaguely ("my thing stopped sending data"). A model can rewrite the question into a few precise search queries; you search each and fuse with RRF. LangChain's `MultiQueryRetriever` did this, but it now lives in **`langchain-classic`** (legacy). Building it explicitly takes ten lines and you control the prompt and the cost.

In [ ]:
# 🧪 Offline: the rewriter chain with a scripted model, and multi-query retrieval.
from langchain_core.messages import AIMessage
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

from mastery.testing import ScriptedChatModel

class SearchQueries(BaseModel):
    """Search queries for a documentation search engine."""
    queries: list[str] = Field(description="2-3 short, specific queries using words likely to appear in product docs")

REWRITE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "Rewrite the user's question into 2-3 short search queries for Orbita Labs product documentation "
               "(manuals, FAQs, policies). Use specific terms: product names, error codes, feature names."),
    ("human", "{question}"),
])

def multi_query_search(question: str, rewriter, k: int = 5) -> tuple[list[str], list]:
    queries = [question] + rewriter.invoke({"question": question}).queries[:3]   # keep the original too
    return queries, rrf(*[hybrid_search(store, bm25, q, k=10) for q in queries], top_n=k)

scripted_rewriter = REWRITE_PROMPT | ScriptedChatModel(responses=[AIMessage("", tool_calls=[{
    "name": "SearchQueries", "id": "q1", "args": {"queries": ["display bridge Wi-Fi connection", "bridge offline 2.4 GHz"]}}])],
    repeat_last=True).with_structured_output(SearchQueries)
queries, docs = multi_query_search("my thing on the wall stopped talking to the internet", scripted_rewriter)
print("queries:", queries)
show("multi", docs)
show("plain", hybrid_search(store, bm25, queries[0], k=5))

**You should now see** the original question plus two rewritten queries, and `faq-wifi.md` in the top 3 of the multi-query results; the plain search on the vague question puts `faq-calibration.md` first.

> 💸 Uses a few thousand input tokens of your free-tier quota (the retrieved chunks are input). Without `GROQ_API_KEY` it prints `⏭️ Skipped`.

In [ ]:
# 💸 Live: a real model rewrites the vague question (one small call, ~300 tokens).
if HAVE_GROQ_KEY:
    from mastery.models import get_model

    llm = get_model("fast", temperature=0, reasoning_effort="low")
    live_rewriter = REWRITE_PROMPT | llm.with_structured_output(SearchQueries)
    for vague in ["my thing on the wall stopped talking to the internet", "can I get money back for the boat one?"]:
        queries, docs = multi_query_search(vague, live_rewriter)
        print("queries:", queries)
        show("multi", docs)
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** (with a key) rewrites such as `"Wi-Fi bridge disconnected"`, `"Nimbus Marine refund"`, `"returns policy"`, and the relevant FAQ / policy files in the results.

Cost: one extra model call and 3–4× the searches per question. Use it for vague, conversational questions; skip it for precise ones (or let an agent decide, 06.07).

---
## Part 4 · Re-ranking with a cross-encoder

Embeddings encode the question and each chunk **separately** (fast, approximate). A **cross-encoder** reads the question and one chunk **together** and scores how well the chunk answers it: slower, more precise. Standard recipe: retrieve 20 candidates cheaply, re-rank, keep the top 3–5.

> 🌐 First run downloads the ~80 MB `Xenova/ms-marco-MiniLM-L-6-v2` cross-encoder (no key, no tokens). Unavailable? The cell uses the offline `OverlapReranker` (query-word coverage) so the lesson continues.

In [ ]:
# 🌐 A local cross-encoder (fastembed, ONNX, CPU), else the offline stand-in.
from mastery.rag import OverlapReranker, cross_encoder_reranker

try:
    reranker = cross_encoder_reranker()               # Xenova/ms-marco-MiniLM-L-6-v2
except Exception as exc:
    print(f"⚠️ Cross-encoder unavailable ({type(exc).__name__}: {str(exc)[:120]}); using OverlapReranker.")
    reranker = OverlapReranker()
print("reranker:", reranker.name)

q = "How do I keep salt from damaging my station at sea?"
candidates = retrieve(store, q, k=20)
show("before", candidates[:3])
reranked = reranker.rerank(q, candidates, top_n=3)
show("after", reranked)
print("rerank scores:", [d.metadata["rerank"] for d in reranked])

**You should now see** `reranker: Xenova/ms-marco-MiniLM-L-6-v2` (or the stand-in), the top 3 before and after re-ranking (the Marine manual's corrosion-care page stays first, now with a clear margin), and the cross-encoder scores (unbounded logits such as `-2.1` vs `-8.1`: only their order and gaps matter).

In [ ]:
# 🧪/🌐 Measure: does re-ranking help? (20 candidates → top k)
SEARCHES_PLUS = SEARCHES | {
    "vector+rerank": lambda q, k: reranker.rerank(q, retrieve(store, q, k=20), top_n=k),
    "hybrid+rerank": lambda q, k: reranker.rerank(q, hybrid_search(store, bm25, q, k=20), top_n=k),
}
report(SEARCHES_PLUS)

**You should now see** five rows. On our run re-ranking **did not help**: same golden scores, a tiny MRR gain for vector search, and it even **pulled hybrid's paraphrase R@1 back from 0.90 to 0.80**. Why? This MiniLM cross-encoder was trained on web-search queries (MS MARCO); our chunks are short and our base retrieval is already good, so there is little to fix and some room to break. On large, noisy corpora with many near-duplicates re-ranking is often the biggest single win, and bigger re-rankers (`BAAI/bge-reranker-base`, Cohere) do better. It costs ~20 cross-encoder passes per question (tens of milliseconds on a CPU for short chunks).

### What just happened?

You ran an **ablation**: add one technique at a time, measure, keep what pays. That's the only honest way to build "advanced RAG": every step adds latency, cost and code.

In [ ]:
# 💸 Paid (optional): Cohere's hosted re-ranker on the same question. Needs COHERE_API_KEY and `pip install langchain-cohere`.
import importlib.util

if HAVE_COHERE_KEY and importlib.util.find_spec("langchain_cohere"):
    from langchain_cohere import CohereRerank

    cohere = CohereRerank(model="rerank-v4.0-fast", top_n=3)
    for d in cohere.compress_documents(documents=candidates, query=q):
        print(f"{d.metadata['relevance_score']:.3f}  {d.metadata['source']}")
else:
    print("⏭️ Skipped: needs COHERE_API_KEY (trial keys are free and rate-limited) and langchain-cohere (optional).")

**You should now see** `⏭️ Skipped` without a key; with one, three sources with relevance scores between 0 and 1.

---
## Part 5 · Which techniques, when?

| Technique | Helps when | Costs | Our corpus |
|---|---|---|---|
| **Hybrid (BM25 + vector, RRF)** | Codes, names, rare terms, mixed queries | ~free (BM25 is cheap) | +0.10 paraphrase R@1 (measured) |
| **Query rewriting** | Vague or conversational questions | +1 model call, 3–4× searches | Big gain on vague questions |
| **Cross-encoder re-ranking** | Many near-duplicate or noisy candidates | ~20 local passes / question (or a paid API) | No gain with MiniLM (measured): skip by default |
| **Metadata filters** (06.05) | You know the product / type / tenant | Free | Precise scoping |

Default recipe for this course's project: **vector or hybrid search**; re-ranking is the project's stretch ablation. Your measurements decide, not blog posts.

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `ModuleNotFoundError: rank_bm25` | `BM25Retriever` (langchain-community) needs it | Use `mastery.rag.BM25`, or `pip install rank_bm25` |
| `ImportError: MultiQueryRetriever` from `langchain.retrievers` | Moved to `langchain-classic` in LangChain 1.x | Build multi-query explicitly (Part 3) |
| Hybrid is worse than vector alone | Duplicate chunks, or BM25 over noisy text (HTML chrome) | Clean the text (06.02); compare on your golden set |
| Re-ranking is slow | Too many or too long candidates | Re-rank 10–20 candidates, chunks ≤ ~500 tokens |
| Cross-encoder download fails on Windows | Proxy/SSL, or path too long | 00.02 Step 4b; short repo path; the offline stand-in keeps the lesson running |
| Scores from different methods compared directly | Different scales | Compare **ranks** (RRF) or evaluate end results |

## ⚠️ Production notes

> ⚠️ **Production note: every retrieval trick is a latency budget item.**
> Measure p50/p95 latency per stage (search, rewrite, re-rank, answer) alongside quality; a 2% recall gain that adds 800 ms may not be worth it for a chat UI.

> ⚠️ **Production note: rewritten queries can drift.**
> Log the rewritten queries with each answer; when an answer is wrong, a bad rewrite is a common culprit. Always keep the original question as one of the queries.

## 🏋️ Exercises

1. **(Easy)** Search `"rev B"` with vector, BM25 and hybrid. Which finds the firmware 4.0 release note?
2. **(Medium)** Change RRF's `k` from 60 to 10 in `hybrid_search` (call `rrf(..., k=10)` yourself). Does the ranking change on the E40 question?
3. **(Medium)** Write a scripted rewriter for "can I get money back for the boat one?" that returns `["Nimbus Marine refund", "returns policy"]`. Does multi-query find `policies/returns.md`?
4. **(Stretch)** Re-rank the **hybrid** top-20 for all golden questions and count how often the top-1 source changes. Are the changes improvements?

<details>
<summary><b>Show solutions</b></summary>

**1.** BM25 and hybrid (exact words "revision B" / "rev"... depending on tokenisation); vector may miss it. Exact identifiers are BM25's strength.

**2.** `rrf(retrieve(store, q, k=20), bm25.search(q, k=20), k=10, top_n=3)`: a smaller `k` gives top ranks more weight; on short lists the order rarely changes.

**3.** `ScriptedChatModel(responses=[AIMessage("", tool_calls=[{"name": "SearchQueries", "id": "q2", "args": {"queries": ["Nimbus Marine refund", "returns policy"]}}])], repeat_last=True)`; yes, `returns.md` should be first or second.

**4.** Loop over in-corpus golden questions, compare `hybrid_search(...)[0]` with `reranker.rerank(q, hybrid_search(..., k=20), top_n=1)[0]`, and check against `expected_sources`. Count improvements vs regressions: that's a mini A/B test.
</details>

## ✅ Checkpoint

- [ ] I can explain when BM25 beats vectors and vice versa, and how RRF fuses them.
- [ ] I built multi-query retrieval explicitly (no `MultiQueryRetriever`).
- [ ] I re-ranked candidates with a local cross-encoder (or the stand-in) and measured the effect.
- [ ] I keep only the techniques that improve my metrics.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Which re-rankers does fastembed support today (`TextCrossEncoder.list_supported_models()`), and is there still no LangChain-core equivalent of `MultiQueryRetriever` outside `langchain-classic`? Log the answers in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- BM25 finds exact terms, vectors find meaning; RRF combines rankings without score calibration.
- Query rewriting helps vague questions at the cost of one model call.
- Cross-encoders re-rank a small candidate set precisely, locally and for free.
- Measure each addition (ablation) before keeping it.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 06.09 · Measuring RAG quality (preview of Module 10)** → [`06-09-rag-evaluation-preview.ipynb`](06-09-rag-evaluation-preview.ipynb).

[Module 06 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)